# 06 Forecasts

This notebook generates probabilistic VAR forecast trajectories using different innovation models.

Innovation models:

- Gaussian
- Bootstrap
- Student-t
- Diffusion

The goal is to propagate sampled future shocks through the fitted VAR recursion and save forecast paths for later calibration evaluation.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.seeds import set_seed
from src.utils.device import get_device

from src.experiments.config import ExperimentConfig
from src.experiments.paths import result_dirs
from src.experiments.artifacts import (
    load_array_npz,
    save_array_npz,
    save_table,
    save_json,
    save_current_figure,
)

from src.api.modeling import fit_model
from src.api.innovations import fit_innovations, sample_innovations
from src.api.forecasts import generate_forecasts

set_seed(123)

config = ExperimentConfig()
device = get_device()

dgp_names = config.dgp_names

forecast_models = [
    "VAR",
    "RNN",
]

innovation_model_names = [
    "gaussian",
    "bootstrap",
    "student_t",
    "diffusion",
]

lags = config.lags
n_train = config.n_train
horizon = config.horizon
n_paths = config.n_paths

rnn_context_length = 40
rnn_prediction_length = horizon
rnn_hidden_dim = 64
rnn_num_layers = 1
rnn_dropout = 0.0
rnn_batch_size = 32
rnn_epochs = 100
rnn_lr = 1e-3
rnn_device = device

dirs = result_dirs(
    "06_forecasts",
    test=False,
)

simulated_dir = ROOT / "data" / "simulated"

datasets = {}

for dgp_name in dgp_names:
    dgp_data = load_array_npz(
        simulated_dir / f"{dgp_name}_var_dgp.npz"
    )

    y = dgp_data["y"]

    y_train = y[:n_train]
    y_test = y[n_train:n_train + horizon]

    datasets[dgp_name] = {
        "y": y,
        "y_train": y_train,
        "y_test": y_test,
    }

    print(
        dgp_name,
        "train:",
        y_train.shape,
        "test:",
        y_test.shape,
    )

In [ ]:
fitted_models = {}

for forecast_model in forecast_models:
    fitted_models[forecast_model] = {}

    for dgp_name, data in datasets.items():

        y_train = data["y_train"]

        if forecast_model == "VAR":
            fit = fit_model(
                data=y_train,
                model="var",
                lags=lags,
                include_intercept=True,
                save=True,
                output_dir=dirs["models"] / "var" / dgp_name,
            )

        elif forecast_model == "RNN":
            fit = fit_model(
                data=y_train,
                model="rnn",
                context_length=rnn_context_length,
                prediction_length=rnn_prediction_length,
                hidden_dim=rnn_hidden_dim,
                num_layers=rnn_num_layers,
                dropout=rnn_dropout,
                batch_size=rnn_batch_size,
                epochs=rnn_epochs,
                lr=rnn_lr,
                device=rnn_device,
                verbose=False,
                save=True,
                output_dir=dirs["models"] / "rnn" / dgp_name,
            )

        fitted_models[forecast_model][dgp_name] = fit

        print(
            forecast_model,
            dgp_name,
            "fit complete",
        )

In [ ]:
innovation_models = {}
innovation_paths = {}

diffusion_dir = ROOT / "results" / "innovations" / "05_diffusion"

for forecast_model in forecast_models:
    innovation_models[forecast_model] = {}
    innovation_paths[forecast_model] = {}

    for dgp_name in dgp_names:
        innovation_models[forecast_model][dgp_name] = {}
        innovation_paths[forecast_model][dgp_name] = {}

        residual_path = (
            ROOT
            / "results"
            / "residuals"
            / "03_residuals"
            / f"{dgp_name}_{forecast_model.lower()}_residuals.npz"
        )

        residual_data = load_array_npz(residual_path)
        residuals = residual_data["residuals"]

        for method in [
            "gaussian",
            "bootstrap",
            "student_t",
        ]:
            kwargs = {}

            if method == "student_t":
                kwargs["df"] = config.student_t_df

            innov_model = fit_innovations(
                residuals=residuals,
                method=method,
                **kwargs,
            )

            shocks = sample_innovations(
                innovation_model=innov_model,
                n_paths=n_paths,
                horizon=horizon,
                seed=config.seed,
            )

            innovation_models[forecast_model][dgp_name][method] = innov_model
            innovation_paths[forecast_model][dgp_name][method] = shocks

            print(
                forecast_model,
                dgp_name,
                method,
                shocks.shape,
            )

        # diffusion was learned in Notebook 05 as flat one-step samples
        diffusion_path = (
            diffusion_dir
            / forecast_model.lower()
            / dgp_name
            / "diffusion_samples.npz"
        )

        diffusion_data = load_array_npz(diffusion_path)
        diffusion_samples = diffusion_data["innovations"]

        k = diffusion_samples.shape[1]
        needed = n_paths * horizon

        if diffusion_samples.shape[0] < needed:
            raise ValueError(
                f"Not enough diffusion samples for {forecast_model} | {dgp_name}. "
                f"Need {needed}, got {diffusion_samples.shape[0]}."
            )

        diffusion_shocks = diffusion_samples[:needed].reshape(
            n_paths,
            horizon,
            k,
        )

        innovation_paths[forecast_model][dgp_name]["diffusion"] = diffusion_shocks

        print(
            forecast_model,
            dgp_name,
            "diffusion",
            diffusion_shocks.shape,
        )

In [ ]:
def generate_rnn_residual_injection_forecasts(
    fitted_model,
    y_history,
    residual_shocks,
):
    """
    RNN mean forecast + sampled residual shocks.

    residual_shocks:
        Shape (n_paths, horizon, k)
    """
    from src.deep_ar.forecast import rnn_predict_distribution

    context_length = fitted_model["context_length"]

    context = y_history[-context_length:]

    mean, scale = rnn_predict_distribution(
        model=fitted_model["model"],
        context=context,
        device=fitted_model.get("device", "cpu"),
    )

    forecast_paths = mean[None, :, :] + residual_shocks

    return {
        "forecast_paths": forecast_paths,
        "innovation_paths": residual_shocks,
        "predictive_mean": mean,
        "predictive_scale": scale,
    }

## Recursive Forecast Generation

Each innovation path is propagated through the fitted VAR model.

For each DGP and innovation model, the forecast recursion is:

$y_{t+h}^{(m)} = c + A y_{t+h-1}^{(m)} + \varepsilon_{t+h}^{(m)}$

where $\varepsilon_{t+h}^{(m)}$ is sampled from one of the fitted innovation models.

In [ ]:
from src.forecasting.monte_carlo import simulate_forecast_paths

forecast_store = {}

for forecast_model in forecast_models:
    forecast_store[forecast_model] = {}

    for dgp_name in dgp_names:
        forecast_store[forecast_model][dgp_name] = {}

        y_history = datasets[dgp_name]["y_train"]

        for innovation_model in innovation_model_names:

            if forecast_model == "VAR":

                shocks = innovation_paths[forecast_model][dgp_name][innovation_model]

                paths = simulate_forecast_paths(
                    y_history=y_history[-lags:],
                    beta=fitted_models[forecast_model][dgp_name]["beta"],
                    innovation_paths=shocks,
                    lags=lags,
                    include_intercept=True,
                )

                forecast = {
                    "forecast_paths": paths,
                    "innovation_paths": shocks,
                    "horizon": horizon,
                    "n_paths": n_paths,
                    "forecast_model": "VAR",
                    "innovation_model": innovation_model,
                    "y_true": datasets[dgp_name]["y_test"],
                }

            elif forecast_model == "RNN":

                shocks = innovation_paths[forecast_model][dgp_name][innovation_model]

                rnn_result = generate_rnn_residual_injection_forecasts(
                    fitted_model=fitted_models[forecast_model][dgp_name],
                    y_history=y_history,
                    residual_shocks=shocks,
                )

                forecast = {
                    "forecast_paths": rnn_result["forecast_paths"],
                    "innovation_paths": rnn_result["innovation_paths"],
                    "predictive_mean": rnn_result["predictive_mean"],
                    "predictive_scale": rnn_result["predictive_scale"],
                    "horizon": horizon,
                    "n_paths": n_paths,
                    "forecast_model": "RNN",
                    "innovation_model": innovation_model,
                    "y_true": datasets[dgp_name]["y_test"],
                }   

                forecast["forecast_model"] = "RNN"
                forecast["innovation_model"] = innovation_model
                forecast["y_true"] = datasets[dgp_name]["y_test"]

            forecast_store[forecast_model][dgp_name][innovation_model] = forecast

            save_array_npz(
                (
                    dirs["forecasts"]
                    / forecast_model.lower()
                    / dgp_name
                    / f"{innovation_model}_forecast_paths.npz"
                ),
                forecast_paths=forecast["forecast_paths"],
                innovation_paths=forecast["innovation_paths"],
                y_true=forecast["y_true"],
            )

            print(
                forecast_model,
                dgp_name,
                innovation_model,
                forecast["forecast_paths"].shape,
            )

In [ ]:
summary_rows = []

for forecast_model in forecast_models:
    for dgp_name in dgp_names:

        y_true = datasets[dgp_name]["y_test"]

        for innovation_model in innovation_model_names:

            paths = forecast_store[
                forecast_model
            ][
                dgp_name
            ][
                innovation_model
            ]["forecast_paths"]

            summary = summarize_probabilistic_forecast(
                forecast_paths=paths,
                y_true=y_true,
                interval=(0.05, 0.95),
                nominal_levels=(0.5, 0.8, 0.9),
            )

            row = make_summary_row(
                dgp_name=dgp_name,
                forecast_model=forecast_model,
                innovation_model=innovation_model,
                summary=summary,
            )

            summary_rows.append(row)

forecast_summary_df = pd.DataFrame(summary_rows)

save_table(
    forecast_summary_df,
    dirs["tables"] / "forecast_summary.csv",
)

forecast_summary_df

In [ ]:
def plot_forecast_grid_for_dgp(
    dgp_name,
    variable=0,
    save=False,
):
    fig, axes = plt.subplots(
        2,
        len(innovation_model_names),
        figsize=(18, 7),
        sharey=True,
    )

    for row, forecast_model in enumerate(forecast_models):
        for col, innovation_model in enumerate(innovation_model_names):

            ax = axes[row, col]

            y_train = datasets[dgp_name]["y_train"]
            y_true = datasets[dgp_name]["y_test"]

            paths = forecast_store[
                forecast_model
            ][
                dgp_name
            ][
                innovation_model
            ]["forecast_paths"]

            lower, upper = prediction_interval(
                paths,
                lower_q=0.05,
                upper_q=0.95,
            )

            median = median_forecast(paths)

            train_tail = y_train[-80:, variable]
            test_values = y_true[:, variable]

            x_train = np.arange(len(train_tail))
            x_forecast = np.arange(
                len(train_tail),
                len(train_tail) + horizon,
            )

            ax.plot(
                x_train,
                train_tail,
                label="train tail",
            )

            ax.plot(
                x_forecast,
                test_values,
                label="true future",
            )

            ax.plot(
                x_forecast,
                median[:, variable],
                linestyle="--",
                label="forecast median",
            )

            ax.fill_between(
                x_forecast,
                lower[:, variable],
                upper[:, variable],
                alpha=0.25,
            )

            if row == 0:
                ax.set_title(innovation_model)

            if col == 0:
                ax.set_ylabel(forecast_model)

    handles, labels = axes[0, 0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.94])

    if save:
        save_current_figure(
            dirs["figures"]
            / f"{dgp_name}_forecast_grid_var_rnn.png"
        )

    plt.show()

In [ ]:
plot_forecast_grid_for_dgp(
    dgp_name="student_t",
    save=True,
)

plot_forecast_grid_for_dgp(
    dgp_name="heteroskedastic",
    save=True,
)

In [ ]:
width_rows = []

for forecast_model in forecast_models:
    for dgp_name in dgp_names:
        for innovation_model in innovation_model_names:

            paths = forecast_store[
                forecast_model
            ][
                dgp_name
            ][
                innovation_model
            ]["forecast_paths"]

            lower, upper = prediction_interval(
                paths,
                lower_q=0.05,
                upper_q=0.95,
            )

            width = average_interval_width(
                lower=lower,
                upper=upper,
            )

            width_rows.append(
                {
                    "forecast_model": forecast_model,
                    "dgp": dgp_name,
                    "innovation_model": innovation_model,
                    "avg_width": float(width.mean()),
                    "width_1": float(width[0]),
                    "width_2": float(width[1]),
                    "width_3": float(width[2]),
                }
            )

width_df = pd.DataFrame(width_rows)

save_table(
    width_df,
    dirs["tables"] / "forecast_interval_widths.csv",
)

width_df

In [ ]:
width_pivot = width_df.pivot_table(
    index=["forecast_model", "dgp"],
    columns="innovation_model",
    values="avg_width",
)

width_pivot

In [ ]:
best_rows = []

for dgp_name in dgp_names:
    subset = forecast_summary_df[
        forecast_summary_df["dgp"] == dgp_name
    ]

    for metric in [
        "energy_score",
        "crps",
        "interval_score",
        "ece",
        "pit_deviation",
    ]:
        best = subset.loc[subset[metric].idxmin()]

        best_rows.append(
            {
                "dgp": dgp_name,
                "metric": metric,
                "best_forecast_model": best["forecast_model"],
                "best_innovation_model": best["innovation_model"],
                "best_value": best[metric],
            }
        )

best_forecast_df = pd.DataFrame(best_rows)

save_table(
    best_forecast_df,
    dirs["tables"] / "best_forecast_models.csv",
)

best_forecast_df

In [ ]:
save_json(
    {
        "notebook": "06_forecasts",
        "forecast_models": forecast_models,
        "dgp_names": dgp_names,
        "innovation_model_names": innovation_model_names,
        "lags": lags,
        "n_train": n_train,
        "horizon": horizon,
        "n_paths": n_paths,
        "forecast_paths_saved": True,
    },
    dirs["logs"] / "forecast_config.json",
)

print(
    "Saved forecasts to:",
    dirs["forecasts"],
)

print(
    "Saved tables to:",
    dirs["tables"],
)

print(
    "06_forecasts completed."
)